## MiRAGE-C: 18 维特征生成 (C-Dataset, 1 疾病 + 7 药物)

In [ ]:
# --- Cell 1: C-Dataset 配置 ---
import pandas as pd
import os
# 数据路径: 自动定位项目根 (兼容从根目录或 code/ 启动)
if os.path.exists("data"):
    DATA_ROOT = "data"
elif os.path.exists("../data"):
    DATA_ROOT = "../data"
else:
    raise FileNotFoundError("找不到 data/ 目录, 请确认工作目录为项目根")
base_path = os.path.join(DATA_ROOT, "C-Dataset", "SimilarityMatrices")
mapping_base = os.path.join(DATA_ROOT, "C-Dataset", "Mapping")



# 1 个疾病相似度 (Phenotype Semantic)
disease_sim_files = {
    'PS': 'DiseasePS.csv'
}

# 7 个药物相似度 (与 DDCD 对齐, 但文件名带 _C 后缀)
drug_sim_files = {
    'Target':           'target_similarity_C.csv',
    'Category':         'category_simialrity_C.csv',
    'Conditions':       'condition_similarity_C.csv',
    'Description':      'description_similarity_C.csv',
    'Mechanism':        'mechanism_similarity_C.csv',
    'Pharmacodynamics': 'pharmacodynamics_similarity_C.csv',
    'Smile':            'SMILE_similarity_C.csv'
}

DISEASE_FEATURE_NAMES = ['PS']            # 1 维
DRUG_FEATURE_NAMES = list(drug_sim_files.keys())  # 7 维
COUNT_FEATURES = ['count_drug', 'count_disease']

q_score_cols = [f'q_score_{n}' for n in DISEASE_FEATURE_NAMES]   # 1
p_score_cols = [f'p_score_{n}' for n in DRUG_FEATURE_NAMES]      # 7
adj_q_cols   = [f'adj_q_score_{n}' for n in DISEASE_FEATURE_NAMES]
adj_p_cols   = [f'adj_p_score_{n}' for n in DRUG_FEATURE_NAMES]

FEATURE_18 = COUNT_FEATURES + q_score_cols + p_score_cols + adj_q_cols + adj_p_cols
assert len(FEATURE_18) == 18, f"Expected 18, got {len(FEATURE_18)}"
print(f"✅ 配置完成. 特征维度: {len(FEATURE_18)} (= 2计数 + 1+7原始 + 1+7交叉)")

In [3]:
# --- Cell 2: 加载 1 个疾病相似度矩阵 ---
disease_sim_dict = {}
for fname, f in disease_sim_files.items():
    path = os.path.join(base_path, f)
    if os.path.exists(path):
        df = pd.read_csv(path, index_col=0)
        disease_sim_dict[fname] = df
        print(f"✅ Disease: {fname} {df.shape}")
    else:
        print(f"❌ 找不到 {f}")

✅ Disease: PS (409, 409)


In [4]:
# --- Cell 3: 加载 7 个药物相似度矩阵 ---
drug_sim_dict = {}
for fname, f in drug_sim_files.items():
    path = os.path.join(base_path, f)
    if os.path.exists(path):
        df = pd.read_csv(path, index_col=0)
        drug_sim_dict[fname] = df
        print(f"✅ Drug: {fname} {df.shape}")
    else:
        print(f"❌ 找不到 {f}")

✅ Drug: Target (663, 663)
✅ Drug: Category (663, 663)
✅ Drug: Conditions (663, 663)
✅ Drug: Description (663, 663)
✅ Drug: Mechanism (663, 663)
✅ Drug: Pharmacodynamics (663, 663)
✅ Drug: Smile (663, 663)


In [5]:
# --- Cell 4: 加载映射 (整数 ID, 防泄漏) ---
# mapping80_C 用整数 (drug 0~662, disease 0~408)
# mapping_C 用 DrugBank 字符串 (drug) + 整数 (disease)
# 整数 drug idx ↔ DrugBank ID: 第 i 个整数 ↔ 排序后第 i 个 DrugBank ID

mapping80_path = os.path.join(mapping_base, "mapping80_C.csv")
mapping_full_path = os.path.join(mapping_base, "mapping_C.csv")

# 训练映射 (整数 ID, 防泄漏)
mapping_train = pd.read_csv(mapping80_path)
mapping_train.columns = ['drug', 'disease']
mapping_train['drug'] = mapping_train['drug'].astype(int)
mapping_train['disease'] = mapping_train['disease'].astype(int)

# 全量映射 (DrugBank + 整数)
mapping_full = pd.read_csv(mapping_full_path)
mapping_full.columns = ['DrugID', 'DiseaseID']
mapping_full['DrugID'] = mapping_full['DrugID'].astype(str).str.strip()
mapping_full['DiseaseID'] = mapping_full['DiseaseID'].astype(int)

# 整数 drug idx → DrugBank 字符串 (查 drug 相似度矩阵用)
sorted_drugbanks = sorted(mapping_full['DrugID'].unique())
int_to_drugbank = {i: db for i, db in enumerate(sorted_drugbanks)}
drugbank_to_int = {db: i for i, db in int_to_drugbank.items()}

print(f"✅ mapping80 (整数 ID): {len(mapping_train):,} 关联")
print(f"✅ mapping_full (混合 ID): {len(mapping_full):,} 关联")
print(f"   drug 范围: {mapping_train['drug'].min()}-{mapping_train['drug'].max()}")
print(f"   disease 范围: {mapping_train['disease'].min()}-{mapping_train['disease'].max()}")
print(f"   整数↔DrugBank: int 0 ↔ {int_to_drugbank[0]}")

✅ mapping80 (整数 ID): 2,026 关联
✅ mapping_full (混合 ID): 2,532 关联
   drug 范围: 0-662
   disease 范围: 0-408
   整数↔DrugBank: int 0 ↔ DB00014


In [ ]:
# --- Cell 5: 核心特征计算 (18 维) ---
# 论文 Eq.3: q_score = max Sim(s', s), p_score = max Sim(d', d)
# 论文 Eq.4: adj = score × |N| (交叉乘法)
# 唯一区别: 疾病侧仅 1 个相似度 (PS), 药物侧 7 个

import numpy as np
from tqdm import tqdm


def normalize_sim_matrix(mat):
    mat = mat.copy()

    def _coerce(x):
        try:
            return int(x)
        except Exception:
            return str(x)

    mat.index = pd.Index([_coerce(x) for x in mat.index], name=mat.index.name)
    mat.columns = pd.Index([_coerce(x) for x in mat.columns], name=mat.columns.name)
    return mat


# 归一化相似度矩阵的行列标签，避免整数 ID 与字符串 ID 混用
for fname, mat in disease_sim_dict.items():
    disease_sim_dict[fname] = normalize_sim_matrix(mat)
for fname, mat in drug_sim_dict.items():
    drug_sim_dict[fname] = normalize_sim_matrix(mat)

# 预计算邻居索引 (整数 ID, 来自 mapping80)
drug_to_diseases_train = mapping_train.groupby('drug')['disease'].apply(set).to_dict()
disease_to_drugs_train = mapping_train.groupby('disease')['drug'].apply(set).to_dict()

# 标签索引: drug_int 在 disease_int 的全量已知药物集合中
disease_to_drugs_full = {}
for _, row in mapping_full.iterrows():
    d_int = drugbank_to_int.get(row['DrugID'])
    if d_int is None: continue
    dis = row['DiseaseID']
    disease_to_drugs_full.setdefault(dis, set()).add(d_int)

all_drugs = sorted(mapping_train['drug'].unique().tolist())
all_diseases = sorted(mapping_train['disease'].unique().tolist())
print(f"药物数: {len(all_drugs):,} | 疾病数: {len(all_diseases):,}")
print(f"总对数: {len(all_drugs)*len(all_diseases):,}")

rows_list = []
for drug_int in tqdm(all_drugs):
    known_diseases = drug_to_diseases_train.get(drug_int, set())
    drug_db = int_to_drugbank[drug_int]

    for disease_int in all_diseases:
        known_drugs = disease_to_drugs_train.get(disease_int, set())
        Ad = known_diseases - {disease_int}
        Bs = known_drugs - {drug_int}

        count_disease = len(Ad)
        count_drug = len(Bs)

        # --- 疾病侧 (1 维) ---
        q_feats = {}
        adj_q_feats = {}
        for fname, mat in disease_sim_dict.items():
            val = 0.0
            if Ad and disease_int in mat.index:
                valid = [d for d in Ad if d in mat.index]
                if valid:
                    v = mat.loc[disease_int, valid].max()
                    if not pd.isna(v):
                        val = float(v)
            q_feats[f'q_score_{fname}'] = val
            adj_q_feats[f'adj_q_score_{fname}'] = val * count_drug

        # --- 药物侧 (7 维) ---
        p_feats = {}
        adj_p_feats = {}
        for fname, mat in drug_sim_dict.items():
            val = 0.0
            if Bs and drug_db in mat.index:
                # 整数 idx → DrugBank 字符串 (查矩阵)
                valid_db = [int_to_drugbank[b] for b in Bs if b in int_to_drugbank]
                valid_db = [d for d in valid_db if d in mat.index]
                if valid_db:
                    v = mat.loc[drug_db, valid_db].max()
                    if not pd.isna(v):
                        val = float(v)
            p_feats[f'p_score_{fname}'] = val
            adj_p_feats[f'adj_p_score_{fname}'] = val * count_disease

        # --- 组装 18 维 ---
        row = {
            'drugID': drug_int,
            'diseaseID': disease_int,
            'count_drug': count_drug,
            'count_disease': count_disease,
        }
        row.update(q_feats)
        row.update(p_feats)
        row.update(adj_q_feats)
        row.update(adj_p_feats)
        row['label'] = 1 if drug_int in disease_to_drugs_full.get(disease_int, set()) else 0

        rows_list.append(row)

df_scores = pd.DataFrame(rows_list).fillna(0.0)
feature_cols = [c for c in df_scores.columns if c not in ['drugID', 'diseaseID', 'label']]
print(f"\n✅ 计算完成! {len(df_scores):,} 行, {len(feature_cols)} 维特征 (预期 18)")
assert len(feature_cols) == 18
print(f"  正样本: {(df_scores['label']==1).sum():,} | 负: {(df_scores['label']==0).sum():,}")
df_scores.head(3)

In [7]:
# --- Cell 6: 保存结果 ---
output_file = "results/MiRAGE_score_C.csv"
os.makedirs("results", exist_ok=True)
df_scores.to_csv(output_file, index=False)

print(f"✅ 结果已保存至 {output_file}")
print(f"   总行数: {len(df_scores):,}")
print(f"   总列数: {len(df_scores.columns)} (= drugID + diseaseID + label + 18特征)")
print(f"   正样本: {(df_scores['label']==1).sum():,}")
print(f"   负样本: {(df_scores['label']==0).sum():,}")
print(f"\n前 5 行预览:")
df_scores.head()

✅ 结果已保存至 results/MiRAGE_score_C.csv
   总行数: 271,167
   总列数: 21 (= drugID + diseaseID + label + 18特征)
   正样本: 2,532
   负样本: 268,635

前 5 行预览:


,drugID,diseaseID,count_drug,count_disease,q_score_PS,p_score_Target,p_score_Category,p_score_Conditions,p_score_Description,p_score_Mechanism,...,p_score_Smile,adj_q_score_PS,adj_p_score_Target,adj_p_score_Category,adj_p_score_Conditions,adj_p_score_Description,adj_p_score_Mechanism,adj_p_score_Pharmacodynamics,adj_p_score_Smile,label
0,0,0,3,4,0.122018,0.0,0.035088,0.000000,0.695154,0.847958,...,0.101754,0.366054,0.0,0.140351,0.000000,2.780617,3.391831,2.718905,0.407018,0
1,0,1,5,4,0.065148,0.0,0.061224,0.000000,0.702717,0.849429,...,0.152174,0.325740,0.0,0.244898,0.000000,2.810869,3.397717,2.442607,0.608696,0
2,0,2,1,4,0.093803,0.0,0.029412,0.000000,0.736795,0.618848,...,0.029091,0.093803,0.0,0.117647,0.000000,2.947178,2.475391,2.708611,0.116364,0
3,0,3,10,4,0.234255,0.0,0.156250,0.333333,0.889292,0.838318,...,0.150820,2.342550,0.0,0.625000,1.333333,3.557169,3.353273,2.690364,0.603279,0
4,0,4,1,4,0.081014,0.0,0.015873,0.000000,0.713334,0.814560,...,0.098182,0.081014,0.0,0.063492,0.000000,2.853336,3.258239,2.246842,0.392727,0
